[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc4_ml/exercices/seance3_exercices.ipynb)

# Séance 4.3 — Arbres et forêts — ce qui fait vraiment la prédiction

**Exercices** · durée : 2h (≈50 min de cours, ≈50 min d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- lire un arbre de décision comme une suite de règles métier
- choisir la complexité d'un modèle par validation croisée, sans toucher au test
- dire pourquoi l'importance native d'un modèle est biaisée
- mesurer l'importance d'une variable par permutation, sur le jeu de test
- montrer le **sens** d'un effet avec une dépendance partielle

## Comment ça marche

La feuille compte **deux parties**, à faire dans l'ordre.

**Partie 1 — l'échauffement.** Le code est déjà écrit, il ne reste que les `____` à
remplir. Chaque exercice se termine par une cellule de **vérification** qui vous dit
immédiatement si votre réponse est bonne.

**Partie 2 — les questions.** Une question, une cellule **vide** : à vous d'écrire le
code entier. Il n'y a pas de vérification automatique — on les corrige ensemble en
séance, et la correction est publiée après.

> ⚠️ Si une vérification de la partie 1 affiche `NameError`, c'est que la cellule
au-dessus n'a pas été exécutée, ou qu'il y reste un `____`. Complétez-la, relancez-la,
puis relancez la vérification.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.metrics import accuracy_score, roc_auc_score

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc4_ml/data/"

In [ ]:
def verifier(nom, condition, indice=""):
    """Affiche un retour immediat sans interrompre le notebook."""
    print("OK   -", nom) if condition else print("A REVOIR -", nom, ":", indice)

Chargement des données utilisées dans toute la feuille :

In [ ]:
tel = pd.read_csv(BASE + "churn.csv")
tel["total"] = pd.to_numeric(tel["total"], errors="coerce")
tel = tel.dropna(subset=["total"])

y = tel["churn"]
# .astype(float) : les dependances partielles refusent les colonnes
# entieres, et les 0/1 de get_dummies sont des booleens
X = pd.get_dummies(tel.drop(columns=["churn"]), drop_first=True).astype(float)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
print(X.shape[1], "variables |", len(X_tr), "abonnes d'apprentissage")

---

# Partie 1 — L'échauffement

Le code est déjà écrit : il ne reste que les `____` à remplir. Allez vite, l'essentiel
de la séance est dans la partie 2.

### Exercice 1 — L'arbre et ses règles

> **Votre mission :**
> - Ajuster un `DecisionTreeClassifier` de profondeur 3 (`random_state=42`) → `a`.
> - Afficher ses règles avec `export_text`.
> - Mettre le nom de la variable de la **première coupure** dans `premiere`.

In [ ]:
a = DecisionTreeClassifier(max_depth=____, random_state=42).fit(X_tr, y_tr)

print(export_text(a, feature_names=list(X.columns)))
premiere = "____"

In [ ]:
verifier("1 - premiere coupure", premiere == "contrat_mensuel",
         "regardez la premiere ligne de export_text")

### Exercice 2 — Noter l'arbre

> **Votre mission :**
> - Calculer la justesse et l'AUC de `a` sur le **test** → `just_a` et `auc_a`, arrondies à 3 décimales.

In [ ]:
just_a = round(accuracy_score(y_te, a.predict(X_te)), 3)
auc_a = round(roc_auc_score(y_te, a.predict_proba(X_te)[:, ____]), 3)

print("justesse", just_a, "| AUC", auc_a)

In [ ]:
verifier("2a - justesse de l'arbre", abs(just_a - 0.785) < 0.02, "accuracy_score")
verifier("2b - AUC de l'arbre", abs(auc_a - 0.813) < 0.02,
         "l'AUC se calcule sur les probabilites, colonne d'indice 1")

### Exercice 3 — La forêt

> **Votre mission :**
> - Ajuster une `RandomForestClassifier` de 200 arbres (`random_state=42`) → `f`.
> - Calculer son AUC de test → `auc_f`, et comparer à celle de l'arbre.

In [ ]:
f = RandomForestClassifier(n_estimators=____, random_state=42).fit(X_tr, y_tr)
auc_f = round(roc_auc_score(y_te, f.predict_proba(X_te)[:, 1]), 3)

print("arbre", auc_a, "| foret", auc_f)

In [ ]:
verifier("3 - AUC de la foret", abs(auc_f - 0.805) < 0.02,
         "n_estimators=200 pour 200 arbres")

### Exercice 4 — La bonne profondeur, sans toucher au test

> **Votre mission :**
> - Par validation croisée à 5 plis sur l'**apprentissage**, comparer les profondeurs 2, 4 et 8.
> - Mettre l'AUC moyenne de la profondeur 4 dans `auc_cv4` (3 décimales).

In [ ]:
for prof in [2, 4, 8]:
    s = cross_val_score(DecisionTreeClassifier(max_depth=prof, random_state=42),
                        ____, ____, cv=5, scoring="roc_auc")
    print("profondeur", prof, ":", round(s.mean(), 3))

auc_cv4 = round(cross_val_score(
    DecisionTreeClassifier(max_depth=4, random_state=42),
    X_tr, y_tr, cv=5, scoring="roc_auc").mean(), 3)

In [ ]:
verifier("4 - AUC croisee a la profondeur 4", abs(auc_cv4 - 0.826) < 0.02,
         "cross_val_score sur X_tr et y_tr, jamais sur le test")

### Exercice 5 — L'importance native

> **Votre mission :**
> - Extraire l'importance native de la forêt dans une `Series` indexée par le nom des colonnes → `native`.
> - Mettre en tête du classement dans `top_native`.

In [ ]:
native = pd.Series(f.____, index=X.columns)

top_native = native.idxmax()
print(native.sort_values(ascending=False).head(4).round(3))

In [ ]:
verifier("5 - variable la plus importante (native)", top_native == "total",
         "l'attribut s'appelle feature_importances_, avec un underscore final")

### Exercice 6 — L'importance par permutation

> **Votre mission :**
> - Mesurer l'importance par permutation sur le **jeu de test** (`n_repeats=5`, `random_state=42`, `scoring='roc_auc'`) → `perm`.
> - Mettre la variable arrivée en tête dans `top_perm`.
> - Comparez à l'exercice 5.

In [ ]:
pi = permutation_importance(f, ____, ____, n_repeats=5,
                            random_state=42, scoring="roc_auc")
perm = pd.Series(pi.importances_mean, index=X.columns)

top_perm = perm.idxmax()
print(perm.sort_values(ascending=False).head(4).round(4))

In [ ]:
verifier("6 - variable la plus importante (permutation)", top_perm == "contrat_mensuel",
         "permutation_importance(f, X_te, y_te, ...)")

### Exercice 7 — Le rang qui s'effondre

> **Votre mission :**
> - Quel est le **rang** de `total` dans chacun des deux classements ?
> - Mettre les deux rangs dans `rang_native` et `rang_perm` (1 = première).

In [ ]:
ordre_native = native.sort_values(ascending=False).index
ordre_perm = perm.sort_values(ascending=False).index

rang_native = list(ordre_native).index("total") + 1
rang_perm = list(____).index("total") + 1
print("total : rang", rang_native, "en natif,", rang_perm, "en permutation")

In [ ]:
verifier("7a - rang natif de total", rang_native == 1, "c'est la premiere du classement natif")
verifier("7b - rang par permutation", rang_perm > 5,
         "cherchez la position de total dans ordre_perm")

### Exercice 8 — Le sens de l'effet

> **Votre mission :**
> - Tracer la dépendance partielle de la forêt pour l'ancienneté (`anc`).
> - Mettre dans `sens` le mot « baisse » ou « monte », selon ce que fait le risque quand l'ancienneté augmente.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
PartialDependenceDisplay.from_estimator(f, X_te, ["____"], ax=ax)
plt.show()

sens = "____"

In [ ]:
verifier("8 - sens de l'effet de l'anciennete", sens == "baisse",
         "regardez la courbe : le risque augmente-t-il avec l'anciennete ?")

### Exercice 9 — L'erreur de jeu

> **Votre mission :**
> - Refaire la permutation sur le jeu d'**apprentissage** (`n_repeats=3`) → `top_appr`.
> - Le résultat diffère-t-il de celui du test ? Laquelle des deux mesures faut-il retenir ?

In [ ]:
pa = permutation_importance(f, X_tr, y_tr, n_repeats=3,
                            random_state=42, scoring="roc_auc")
top_appr = pd.Series(pa.importances_mean, index=X.columns).____()

print("sur apprentissage :", top_appr, "| sur test :", top_perm)

In [ ]:
verifier("9 - importance mesuree sur l'apprentissage", isinstance(top_appr, str),
         "idxmax() donne le nom de la variable en tete")

### Exercice 10 — Question de synthèse

> **Votre mission :**
> - Le comité veut **une** action, chiffrée.
> - Calculer le taux de départ des contrats mensuels et celui des engagements deux ans → `t_mensuel` et `t_deux_ans` (en %, 1 décimale).
> - En déduire l'écart en points → `ecart_points`. Puis rédigez la recommandation en commentaire.

In [ ]:
taux = (tel.groupby("contrat")["churn"].mean() * 100).round(1)

t_mensuel = taux["mensuel"]
t_deux_ans = taux["____"]
ecart_points = round(t_mensuel - t_deux_ans, 1)
print(t_mensuel, "% contre", t_deux_ans, "% ->", ecart_points, "points")

In [ ]:
verifier("10a - churn des contrats mensuels", t_mensuel == 42.7, "groupby('contrat')")
verifier("10b - ecart en points", abs(ecart_points - 39.9) < 0.2,
         "la modalite s'ecrit deux_ans")

---

# Partie 2 — Les questions

Ici, plus de trous : **la cellule sous chaque question est vide**, et c'est à vous
d'écrire le code en entier. C'est exactement ce qu'on vous demandera pour le projet
final, et ce que fait un analyste devant un fichier qu'il découvre.

Certaines questions utilisent une commande que le cours n'a pas montrée. Quand c'est le
cas, l'énoncé vous la donne — savoir se servir d'une commande qu'on vient de lire fait
partie du métier.

> 💡 Pas de vérification automatique dans cette partie. Affichez systématiquement votre
> résultat, et demandez-vous s'il est **plausible** avant de passer à la suite : c'est
> le seul contrôle dont vous disposerez en entreprise.

### Question 11 — Les deux classements côte à côte

> **Votre mission :**
> - Construire un tableau à trois colonnes : l'importance native, l'importance par permutation, et le rang dans chacune.
> - Trier par permutation décroissante. Quelles variables changent le plus de place ?

### Question 12 — Fabriquer le biais soi-même

> **Votre mission :**
> - Ajouter à `X` une colonne `bruit` remplie de nombres **au hasard**, sans aucun lien avec la cible.
> - Réajuster la forêt et regarder où cette colonne se classe dans chacune des deux importances.
> - *Nouveau :* `np.random.default_rng(42).random(len(X))` fabrique des nombres au hasard.

### Question 13 — Deux variables qui disent la même chose

> **Votre mission :**
> - Ajouter une colonne `anc_bis`, copie exacte de `anc`. Réajuster la forêt et mesurer la permutation.
> - Que devient l'importance de `anc` ? Pourquoi ?
> - Quelle précaution en tirer avant de lire un classement d'importances ?

### Question 14 — La dépendance partielle du contrat

> **Votre mission :**
> - Tracer la dépendance partielle pour `contrat_mensuel`, puis pour `mensuel` (la facture).
> - Chiffrer l'écart de probabilité prédite entre les deux extrémités de chaque courbe.
> - Laquelle des deux variables offre un levier d'action réel ?

### Question 15 — Combien d'arbres faut-il ?

> **Votre mission :**
> - Comparer l'AUC de test d'une forêt à 5, 20, 100 et 400 arbres.
> - À partir de combien le gain devient-il négligeable ?
> - Que coûte le passage de 100 à 400 ?

### Question 16 — Un arbre par segment

> **Votre mission :**
> - Ajuster un arbre de profondeur 3 sur les seuls abonnés au contrat **mensuel**.
> - Ses règles sont-elles les mêmes que celles de l'arbre général ?
> - Qu'est-ce que ça dit de l'idée d'un modèle unique pour toute la clientèle ?

### Question 17 — La validation croisée est bruitée

> **Votre mission :**
> - Afficher les cinq scores individuels de la validation croisée à la profondeur 4, et leur écart-type.
> - L'écart entre la profondeur 4 et la profondeur 5 est-il plus grand que ce bruit ?

### Question 18 — Ce que le modèle rate

> **Votre mission :**
> - Isoler les abonnés que la forêt classe « reste » alors qu'ils sont partis.
> - Comparer leur profil à celui des partants correctement détectés.
> - Qu'ont de particulier ceux qu'on ne voit pas venir ?

### Question 19 — Expliquer UN client, avec SHAP

> **Votre mission :**
> - Les importances précédentes sont **globales** : elles décrivent le modèle, pas un client.
> - SHAP décompose une prédiction individuelle en contributions chiffrées.
> - Le paquet n'est pas fourni par Colab : la ligne d'installation vous est donnée, le reste est à écrire.
> - ⚠️ Environ **42 Mo** téléchargés par la machine virtuelle Colab — rien ne passe par votre connexion — et une trentaine de secondes de calcul.
> - Expliquer la prédiction du **premier abonné du jeu de test** : quelles variables la poussent à la hausse, lesquelles à la baisse ?

In [ ]:
!pip install -q shap
import shap

# A vous : expliquer la prediction du premier abonne de X_te

### Question 20 — Question de synthèse

> **Votre mission :**
> - Rédigez la note de direction qui répond à la question du comité : *« qu'est-ce qui fait partir nos clients ? »*
> - Contrainte : trois facteurs classés, chacun avec son **sens** et son ordre de grandeur, plus une phrase sur ce que ces données ne permettent **pas** d'affirmer.
> - Calculez d'abord les chiffres dont vous avez besoin.